In [1]:
# Load the autoreload extension
%load_ext autoreload

# Set autoreload mode
%autoreload 2

# Scraping Analysis & Discussion

In [2]:
from urllib.parse import urljoin, urlparse

import boto3
import requests
from bs4 import BeautifulSoup
import pandas as pd 

def check_s3_prefix_exists(bucket_name, s3_prefix, source_id, specific_file = "annotations.csv"):
    s3 = boto3.client("s3")
    prefix = f"{s3_prefix}/s{source_id}/{specific_file}"

    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix, MaxKeys=1)

    if "Contents" in response:
        # print(f"Prefix exists: {prefix}")
        return True
    else:
        # print(f"Prefix does not exist: {prefix}")
        return False

### Get all publicly available CoralNet sources

In [3]:
url = "https://coralnet.ucsd.edu/source/about/"

resp = requests.get(url, timeout=50)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")
anchors = soup.find_all("a", href=True)

links = sorted(
    {
        urljoin(url, a["href"])
        for a in anchors
        if urlparse(urljoin(url, a["href"])).scheme in ("http", "https")
    }
)

print("Found", len(links), "links on the page.")
source_links = [link for link in links if "/source/" in link]
print("Found", len(source_links), "links on the page.")
all_coralnet_sources = sorted({int(link.split("/")[-2]) for link in source_links})

Found 1976 links on the page.
Found 1966 links on the page.


### Get all scraped CoralNet sources

In [4]:
# Configuration - Where to save the downloaded CoralNet images
bucket_name = "dev-datamermaid-sm-sources"
prefix = "coralnet-public-images"

In [5]:
# data = [] 
# for i, source_id in tqdm.tqdm(enumerate(all_coralnet_sources)):
#     # print(i, "Source ID", source_id)
#     labelset_flag = check_s3_prefix_exists(
#         bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="labelset.csv"
#     )
#     metadata_flag = check_s3_prefix_exists(
#         bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="metadata.csv"
#     )
#     annotations_flag = check_s3_prefix_exists(
#         bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="annotations.csv"
#     )
#     image_list_flag = check_s3_prefix_exists(
#         bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
#     )
#     data.append([i, source_id, labelset_flag, metadata_flag, annotations_flag, image_list_flag]) 

# source_status = pd.DataFrame(data, columns = ["idx", "source_id", "labelset", "metadata", "annotations", "image_list"])
# source_status.to_csv("dataframes/coralnet_source_status.csv")
source_status = pd.read_csv("dataframes/coralnet_source_status.csv")

In [6]:
source_status.head()

,idx,source_id,labelset,metadata,annotations,image_list
0,0,23,True,True,True,True
1,1,57,True,True,True,True
2,2,69,True,False,True,True
3,3,70,True,True,True,True
4,4,82,True,False,True,True


In [7]:
print("The number of unique sources that have at least some scraped data equals", 
      (source_status["labelset"]+source_status["metadata"]+source_status["annotations"]+source_status["image_list"]).sum())

The number of unique sources that have at least some scraped data equals 1498


In [8]:
print("The number of unique sources with specific scraped components:")
print(source_status[["labelset", "metadata", "annotations", "image_list"]].sum())

The number of unique sources with specific scraped components:
labelset       1457
metadata        974
annotations    1452
image_list     1452
dtype: int64


As of the 6th of April 2026, there are 1963 sources on the CoralNet website. The scraping that occurred last time (September-November), resulted in 1498 sources, meaning that since then there have been few hundred new sources that were added or turned public. 

**We need to think of a way to programatically scrape the new sources and add them to the S3 bucket.** 

### Get the information for a specific source
The main components that are necessary for each source are the annotations and image_list (and optionally, the labelset and metadata). 

In [9]:
s3 = boto3.client("s3")

source_id = 23

source_components = []
for component in ["labelset.csv", "metadata.csv", "annotations.csv", "image_list.csv"]:
    flag = check_s3_prefix_exists(
        bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file=component
    )
    if flag:
        s3_key_image = f"{prefix}/s{source_id}/{component}"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
        source_components.append(pd.read_csv(obj_il["Body"]))
    else:
        source_components.append(None)
labelset_df, metadata_df, annotations_df, image_list_df = source_components

images_prefix = f"{prefix}/s{source_id}/images/"
paginator = s3.get_paginator("list_objects_v2")
file_keys = []

for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            file_keys.append(key)

The labelset contains the label mapping between IDs and names used within the CoralNet source. It is acquired by getting the CSV file found at the source's labelset link (e.g. https://coralnet.ucsd.edu/source/23/labelset/). 

However, since label IDs are consistent across sources, as long as we have an appropriate mapping between the label ID and the MERMAID labels for each label of interest (as we do in our current version), it isn't a necessity to scrape (and update) for each source.

In [10]:
labelset_df.head()

,Label ID,Name,Short Code
0,108,Meandroid (brain) corals,Brain
1,100,Other scleractinians,HC_other
2,109,Agariciidae,Lettuce
3,110,Montastraea cavernosa,MCAV
4,112,Montastrea franksi,MFRA


The metadata contains the performance for different CoralNet classifiers trained on the source, therefore shouldn't be of much interest and do not need to be downloaded.

In [11]:
metadata_df.head()

,Classifier nbr,Accuracy,Trained on,Date,Traintime,Global id
0,1,67,20,0:00:06,"Nov. 21, 2016",21
1,2,76,408,0:02:32,"Nov. 27, 2016",739
2,3,78,750,0:09:24,"Dec. 5, 2016",1356


The annotations file contains the point annotations for a given source, such that each row in the file represents one annotated point with a specific label (Label ID), at a specific pixel's location (Row, Column) for a given image (Name). The annotations file can also contain a lot of metadata, such as the date, location and depth of the observation which can be very important, though this information is source specific.

The problem with scraping the annotations is that the file is requested as a form on the source images website (e.g. https://coralnet.ucsd.edu/source/23/browse/images/ - you have to be logged in to see the form). The requested form can take a very long time for specific sources (>10000 images) and takes even longer if the image metadata is also requested (in the past I had to manually download such source annotations files and upload them to S3, because the scraping srcipt kept getting stuck). Furthermore, since the annotations file is downloaded as a full csv file, even if only a few new annotations have been added, we have to redownload the entire file from scratch (which combined with the above problem, makes this very tedious). 

In [12]:
annotations_df[["Name", "Row", "Column", "Label ID"]].head()

,Name,Row,Column,Label ID
0,IMG_3401.JPG,735,1008,112
1,IMG_3401.JPG,663,1682,106
2,IMG_3401.JPG,955,1737,106
3,IMG_3401.JPG,1034,1431,105
4,IMG_3401.JPG,851,2036,106


In [13]:
annotations_df.head()

,Name,Date,Site,Depth,Transect,Metermark,Aux5,Height (cm),Latitude,Longitude,...,Strobes,Framing gear used,White balance card,Comments,Row,Column,Label code,Label ID,Annotator,Date annotated
0,IMG_3401.JPG,2012-10-07,PtaCaracol,NaN,A,5,NaN,82,NaN,NaN,...,NaN,NaN,NaN,NaN,735,1008,MFRA,112,bneal,2012-04-16 11:14:48+00:00
1,IMG_3401.JPG,2012-10-07,PtaCaracol,NaN,A,5,NaN,82,NaN,NaN,...,NaN,NaN,NaN,NaN,663,1682,Mix,106,bneal,2012-04-16 11:14:48+00:00
2,IMG_3401.JPG,2012-10-07,PtaCaracol,NaN,A,5,NaN,82,NaN,NaN,...,NaN,NaN,NaN,NaN,955,1737,Mix,106,bneal,2012-04-16 11:14:48+00:00
3,IMG_3401.JPG,2012-10-07,PtaCaracol,NaN,A,5,NaN,82,NaN,NaN,...,NaN,NaN,NaN,NaN,1034,1431,Dead,105,bneal,2012-04-16 11:14:48+00:00
4,IMG_3401.JPG,2012-10-07,PtaCaracol,NaN,A,5,NaN,82,NaN,NaN,...,NaN,NaN,NaN,NaN,851,2036,Mix,106,bneal,2012-04-16 11:14:48+00:00


The image_list contains the name of each image, the image page (link) where it could be accessed and potentially the Image URL (used to download the image). The images are saved using the number of the image (as found on the image page link), rather than using the name of the image, which is why the image_list is required to provide a mapping between the annotations file and the image_list. All of the images are downloaded in the source/images/ subfolder in the S3 bucket.

In the future we can save the images by their name directly to speed up that process (even though it is only done as a postprocessing step once), however, we still need to download the image_list in order to do so and in order to allow us to download future images. 

Regarding the scraping, both the image_list and the images themselves are acquired by going through the different pages of the source/images website (e.g. https://coralnet.ucsd.edu/source/23/browse/images/), therefore it is relatively easy to just update each source with the newly added images. 

In [14]:
image_list_df.head()

,Name,Image Page,Image URL
0,IMG_3401.JPG - Confirmed,/image/12805/view/,https://coralnet-production.s3.us-west-2.amazo...
1,IMG_3402.JPG - Confirmed,/image/12806/view/,https://coralnet-production.s3.us-west-2.amazo...
2,IMG_3403.JPG - Confirmed,/image/12807/view/,https://coralnet-production.s3.amazonaws.com/m...
3,IMG_3404.JPG - Confirmed,/image/12808/view/,https://coralnet-production.s3.us-west-2.amazo...
4,IMG_3407.JPG - Confirmed,/image/12809/view/,https://coralnet-production.s3.amazonaws.com/m...


In [15]:
file_keys[:5]

['coralnet-public-images/s23/images/12755.jpg',
 'coralnet-public-images/s23/images/12756.jpg',
 'coralnet-public-images/s23/images/12757.jpg',
 'coralnet-public-images/s23/images/12758.jpg',
 'coralnet-public-images/s23/images/12759.jpg']

### Source BUG
There may be a discrepancy between the list (and number) of images in the annotation file, image_list and image folder. 
- There was a bug in the last scraping which only saved the a portion of the image_list file for big files that needs to be fixed and all of the image_list files need to be redownloaded.
- The other discrepancies are usually a few images if the image list was updated in the meantime or could potentially be caused by the S3 file system malfunction resulting in copies. 

In [16]:
# s3 = boto3.client("s3")

# counts_list = []
# for i, source_id in tqdm.tqdm(enumerate(all_coralnet_sources)):

#     annotations_flag = check_s3_prefix_exists(
#         bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="annotations.csv"
#     )
#     image_list_flag = check_s3_prefix_exists(
#         bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
#     )


#     if image_list_flag:
#         s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
#         obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
#         image_list_df = pd.read_csv(obj_il["Body"])
#         image_list_count = image_list_df.shape[0]
#     else:
#         image_list_count = 0

#     if annotations_flag:
#         s3_key_annotations = f"{prefix}/s{source_id}/annotations.csv"
#         obj_ann = s3.get_object(Bucket=bucket_name, Key=s3_key_annotations)
#         annotations_df = pd.read_csv(obj_ann["Body"])
#         annotations_count = annotations_df["Name"].nunique()
#     else:
#         annotations_count = 0

#     if image_list_flag:
#         images_prefix = f"{prefix}/s{source_id}/images/"

#         paginator = s3.get_paginator("list_objects_v2")
#         file_count = 0

#         for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
#             for obj in page.get("Contents", []):
#                 key = obj["Key"]
#                 if key != images_prefix and not key.endswith("/"):
#                     file_count += 1
#     else:
#         file_count = 0
#     counts_list.append((source_id, image_list_count, annotations_count, file_count))

# df_counts = pd.DataFrame(counts_list, columns = ["source_id", "image_list_count", "annotations_count", "images_folder_count"])
# df_counts.to_csv("dataframes/coralnet_source_counts.csv", index=False)

df_counts = pd.read_csv("dataframes/coralnet_source_counts.csv")

In [17]:
df_counts

,source_id,image_list_count,annotations_count,images_folder_count
0,23,750,750,750
1,57,1681,1681,1681
2,69,100,100,100
3,70,300,300,300
4,82,1,0,1
...,...,...,...,...
1956,8277,0,0,0
1957,8278,0,0,0
1958,8284,0,0,0
1959,8288,0,0,0


In [18]:
df_counts[(df_counts["image_list_count"]!=df_counts["annotations_count"])+(df_counts["image_list_count"]!=df_counts["images_folder_count"])]

,source_id,image_list_count,annotations_count,images_folder_count
4,82,1,0,1
12,295,1064,79064,79062
16,372,98,66098,66098
17,373,2485,29485,29485
19,428,28,0,28
...,...,...,...,...
1688,7461,21,0,21
1689,7462,21,9,21
1692,7486,100,0,100
1694,7494,27,1,27


In [ ]:
s3 = boto3.client("s3")

source_id = 23

annotations_flag = check_s3_prefix_exists(
    bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="annotations.csv"
)
image_list_flag = check_s3_prefix_exists(
    bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
)


if image_list_flag:
    s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
    obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
    image_list_df = pd.read_csv(obj_il["Body"])

if annotations_flag:
    s3_key_annotations = f"{prefix}/s{source_id}/annotations.csv"
    obj_ann = s3.get_object(Bucket=bucket_name, Key=s3_key_annotations)
    annotations_df = pd.read_csv(obj_ann["Body"])
    
images_prefix = f"{prefix}/s{source_id}/images/"

paginator = s3.get_paginator("list_objects_v2")
file_keys = []

for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            file_keys.append(key)

In [ ]:
df_tmp = df_counts[(df_counts["image_list_count"]!=df_counts["annotations_count"])+(df_counts["image_list_count"]!=df_counts["images_folder_count"])]
df_tmp["images_folder_count"].sum()-df_tmp["image_list_count"].sum()

In [ ]:
s3 = boto3.client("s3")
images_prefix = f"{prefix}/s{source_id}/images/"

paginator = s3.get_paginator("list_objects_v2")
file_count = 0

for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            file_count += 1

print(f"s3://{bucket_name}/{images_prefix} -> {file_count} files")

In [ ]:
image_list_df.shape, annotations_df["Name"].nunique(), file_count

In [ ]:
image_list_df[:10]